# 1. What is actually in a checkpoint

A filename is a claim, not a fact. `RBPNet_21M_20241126_Penalty-5.0.pt` says lambda is
5.0, but the file itself carries `penalty = None`, so that number survives only in the
name. This notebook reads what a model really contains and checks it.

You will see how to:

1. load a model from the `safetensors` + `config.json` pair
2. read its architecture, its provenance and its reference measurements
3. get task names, which live with the dataset and not in the model
4. verify the three things that silently go wrong: eval mode, determinism, and a config
   that disagrees with the weights

In [1]:
import torch

# On a shared machine torch otherwise claims one thread per core and four users
# oversubscribe it. Measured: 34.3 steps/s each with this cap, 3.9 without.
torch.set_num_threads(2)

import json
import parnet_tutorials as pt

print('project root:', pt.project_root())
print('paths config:', pt.config_path())

project root: /data/projects/lambosaur/parnet-tutorials
paths config: /data/projects/lambosaur/parnet-tutorials/config/filepaths.yaml


## Load

`model_path` returns a STEM. The loader appends `.safetensors` for the weights and
`.config.json` for the architecture. If either is missing, you get an error naming the
config key and the resolved path, rather than a failure three cells later.

In [2]:
from parnet.models import RBPNet

stem = pt.model_path('parnet-21m-penalty5')
print('stem:', stem)

model = RBPNet.from_pretrained(stem)
print('parameters:', sum(p.numel() for p in model.parameters()))

stem: /data/parnet_dataset_cache/exported_models/parnet-21m-penalty5


/home/user/projects/parnet-project/parnet--analyses--interpretation/.pixi/envs/parnet-dev-cu12/lib/python3.10/site-packages/torch/nn/modules/conv.py:370: UserWarning: Using padding='same' with even kernel lengths and odd dilation may require a zero-padded copy of the input be created (Triggered internally at /pytorch/aten/src/ATen/native/Convolution.cpp:1025.)
  return F.conv1d(


parameters: 21814495


## What the config says

Everything in the `architecture` block was read off the model at export time rather than
passed in, so it cannot describe a model other than the one it ships with.

The entries that are not shapes matter most. An activation is not a parameter, so a
config that omits it produces a model that loads without error and computes something
different. That is not hypothetical: the 3M backbones below were trained with GELU, and
an earlier export did not record it.

In [3]:
config = model.parnet_config
arch = config['architecture']

for key in ['class', 'n_tasks', 'n_parameters', 'body_width', 'embedding_dim',
            'use_projection', 'head_class', 'mix_coeff_class', 'mix_coeff_range',
            'use_maximum_target_control_logprob', 'penalty_class', 'penalty_factor']:
    print(f'{key:38s} {arch.get(key)}')

print()
print('stem :', arch['stem'])
print('body :', {k: v for k, v in arch['body'].items() if k != 'dilations'})
print('dilations:', arch['body']['dilations'])

class                                  RBPNet
n_tasks                                223
n_parameters                           21814495
body_width                             768
embedding_dim                          768
use_projection                         False
head_class                             TargetControlMix
mix_coeff_class                        PooledLogitMixCoeff
mix_coeff_range                        logit
use_maximum_target_control_logprob     False
penalty_class                          None
penalty_factor                         None

stem : {'class': 'StemConv1D', 'kernel_size': 12, 'activation': 'ReLU', 'dropout': None}
body : {'class': 'ResConvBlock1D', 'kernel_size': 3, 'activation': 'ReLU', 'dropout': 0.25, 'residual': True}
dilations: [1, 1, 3, 5, 9, 16, 28, 50, 87, 153, 269, 471]


### Read the provenance, and separate it from what was measured

`architecture` describes what the file contains. The released 21M carries `penalty = None`,
so `penalty_class` and `penalty_factor` are null, which is the honest description even
though the model was trained at lambda = 5.0.

That lambda is real knowledge, but it is a claim about the training run rather than
something measurable from the weights, so it lives in its own block. The measurable
consequence of the penalty is the reference mean target weight, further down.

In [4]:
print(json.dumps(config['export'], indent=1))

print()
print('penalty_class  :', arch['penalty_class'])
print('penalty_factor :', arch['penalty_factor'], '  <- not in the file')

for claim in config.get('unverifiable_claims', []):
    print()
    print(f"CLAIM: {claim['quantity']} = {claim['value']}, source: {claim['source']}")
    print(f"  recoverable from weights: {claim['recoverable_from_weights']}")
    print(' ', claim['note'])

{
 "written_on": "2026-10-28",
 "format": "safetensors state_dict; architecture in this file",
 "exported_in_eval_mode": true,
 "exported_on": "2026-10-27",
 "source_path": "/data/projects/parnet-project/parnet--analyses--interpretation/resources/models/RBPNet_21M_20241126_Penalty-5.0.pt",
 "source_sha256": "579a0ac3a8efcf15b8175de36c935a5180ddea248c407a7f4587926e392e93b8",
 "source_was_pickled_in_training_mode": false,
 "source_submodules_in_training_mode": 0,
 "eval_mode_is_deterministic": true,
 "config_regenerated_on": "2026-10-28",
 "config_regeneration_reason": "Mixing classes renamed: AdditiveMix -> TargetControlMix, NewAdditiveMix -> ConvexTargetControlMix, SequenceLinearMix -> PooledLogitMixCoeff, MixCoeffMLP -> PooledProbMixCoeff, NewMixCoeffMLP -> PooledProbMixCoeffV2; lambda_anchor -> anchor_strength. Weights are untouched and outputs are bitwise identical."
}

penalty_class  : None
penalty_factor : None   <- not in the file

CLAIM: penalty factor (lambda) = 5.0, source: fi

## Task names belong to the dataset

The model knows it has 223 outputs. It does not know what they are called, and it should
not: the same architecture trained on another dataset has different tracks. Names live in
`config/tasks/`.

Filter by cell type on the `ct` field, never by splitting the name. Content and model
label disagree at 10 of the 223 tracks, and at 4 of those the cell type itself differs.

In [5]:
names = pt.task_names('encode-223')
print('task names:', len(names), '| model outputs:', arch['n_tasks'])
assert len(names) == arch['n_tasks'], 'task config does not match this model'

print('first five:', names[:5])

hepg2 = pt.task_names('encode-hepg2-103')
k562 = pt.task_names('encode-k562-120')
print(f'HepG2 {len(hepg2)} + K562 {len(k562)} = {len(hepg2) + len(k562)}')

# The four tracks whose content cell type differs from the label the models carry.
for idx in (19, 20, 150, 151):
    print(f'  index {idx:3d}: {names[idx]}')

task names: 223 | model outputs: 223
first five: ['AARS_K562', 'AATF_K562', 'ABCF1_K562', 'AGGF1_HepG2', 'AGGF1_K562']
HepG2 103 + K562 120 = 223
  index  19: CSTF2T_K562
  index  20: CSTF2_HepG2
  index 150: SAFB2_K562
  index 151: SAFB_HepG2


## Three checks worth making every time

**Eval mode.** A model pickled in training mode uses batch statistics instead of running
statistics and leaves dropout active, so predictions are both wrong and different on
every call. `from_pretrained` returns eval mode; this asserts it.

**Determinism.** The direct consequence, and a one-line test.

**Config against weights.** `use_projection` is derived metadata. The weights decide, and
loading recomputes it rather than trusting the config.

In [6]:
assert model.training is False
assert all(not m.training for m in model.modules())
print('eval mode: ok')

x = torch.rand(2, 4, 600)
with torch.no_grad():
    assert torch.equal(model(x)['total'], model(x)['total'])
print('deterministic: ok')

from parnet.pretrained import state_dict_has_projection

weights_say = state_dict_has_projection(model.state_dict())
print(f"config says use_projection={arch['use_projection']}, weights say {weights_say}")
assert weights_say == arch['use_projection']
print('config agrees with weights: ok')

eval mode: ok


deterministic: ok
config says use_projection=False, weights say False
config agrees with weights: ok


## The reference measurement

The export stores the mean target weight `w` on a fixed, seeded input set. That single
scalar is sensitive at once to a wrong formula flag, a double sigmoid, a lost penalty and
a reduction mismatch, which is why it is worth carrying in the file.

Reproducing it is the cheapest possible check that a model is the one you think it is.

In [7]:
import random
from parnet.utils import sequence_to_onehot

ref = config['reference']
print(json.dumps({k: v for k, v in ref.items() if k != 'note'}, indent=1))

random.seed(ref['seed'])
seqs = [''.join(random.choice('ACGT') for _ in range(ref['sequence_length']))
        for _ in range(ref['n_sequences'])]
X = torch.stack([sequence_to_onehot(s) for s in seqs]).float()

with torch.no_grad():
    out = model(X)

raw = out['mix_coeff']
w = torch.sigmoid(raw) if arch['mix_coeff_range'] == 'logit' else raw
print()
print(f"stored mean w   {ref['mean_target_weight_w']:.6f}")
print(f"recomputed      {w.mean().item():.6f}")
assert abs(w.mean().item() - ref['mean_target_weight_w']) < 1e-4
print('reference reproduced: ok')

{
 "seed": 20261027,
 "n_sequences": 8,
 "sequence_length": 600,
 "alphabet": "ACGT, generated with random.choice under the stated seed",
 "mean_mix_coeff_raw": 2.277823,
 "mean_target_weight_w": 0.282869,
 "total_mean": -1.747007,
 "total_std": 52.190201
}



stored mean w   0.282869
recomputed      0.282869
reference reproduced: ok


## Compare two models by what they contain

`parnet.21m-none` and the lambda=5.0 model differ only in how hard the penalty pushed the
mixing coefficient down. Neither file records lambda. The measurement does.

Here the same reading is applied to a 3M backbone, which differs from the 21M on every
axis that matters: width, task count, activation, and which mixing rule it uses.

In [8]:
rows = []
for key in ['parnet-21m-penalty5', 'parnet-3m-pretrain-E2']:
    m = RBPNet.from_pretrained(pt.model_path(key))
    a = m.parnet_config['architecture']
    rows.append({
        'model': key,
        'tasks': a['n_tasks'],
        'body_width': a['body_width'],
        'activation': a['body']['activation'],
        'mix_coeff': a['mix_coeff_class'],
        'range': a['mix_coeff_range'],
        'convex_rule': a['use_maximum_target_control_logprob'],
        'mean_w': m.parnet_config['reference']['mean_target_weight_w'],
    })

import pandas as pd
pd.set_option('display.width', 160)
print(pd.DataFrame(rows).to_string(index=False))

                model  tasks  body_width activation           mix_coeff range  convex_rule   mean_w
  parnet-21m-penalty5    223         768       ReLU PooledLogitMixCoeff logit        False 0.282869
parnet-3m-pretrain-E2     51         320       GELU  PooledProbMixCoeff  unit         True 0.538940


The two models use **different mixing rules**, not merely different coefficients:

- the 21M emits an unbounded logit and adds it in log space
- the 3M emits a sigmoid-bounded coefficient and takes a convex combination

Notebook 4 shows why those are not interchangeable, and what goes wrong when the two are
crossed.

## What to take away

1. Read `config.json`, not the filename.
2. Task names come from the dataset, and the cell type comes from the `ct` field.
3. Assert eval mode, determinism, and the reference `w` before trusting any prediction.